# **ETAPA ETL**

Resumo para o professor:

Vamos Padronizar as tabelas

1.   Api IBGE


*   Filtrar a coluna Ano para 2022, considerando que essa é a mesma data de publicação utilizada na base da IBM.
*   Excluir os cargos que não possuem correspondência com a tabela da IBM.
*   Criar uma coluna com os cargos disponíveis em inglês, permitindo o pareamento entre as duas bases e a padronização das nomenclaturas de cargos (conforme tabela IBM).
*   Criar a coluna Salário em Dólar (USD).

2. Tabela IBM

*   Criar as colunas Faixa Etária e Faixa Salarial.
*   excluir coluna Over18 e verificar outras colunas que podemos excluir

3. Tabelas Ranking - CUSTO DO TRABALHO e Ranking - SALÁRIO

*   Somente importar as tabelas no codigo


In [2]:
pip install requests pandas sqlalchemy pymysql openpyxl kagglehub

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------- ----- 1.8/2.2 MB 8.9 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests
import pandas as pd

url = "https://apisidra.ibge.gov.br/values/t/5444/n3/all/v/5932/p/last%201/c694/all"

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()

dados = resposta.json()

# A primeira linha contém a descrição das colunas
descricao_colunas = dados[0]

# Cria o DataFrame com os dados
df = pd.DataFrame(dados[1:])

# Renomeia as colunas
df = df.rename(columns=descricao_colunas)

# Seleciona as colunas necessárias
df = df[
    [
        "Unidade da Federação",
        "Grupamentos ocupacionais no trabalho principal - PNADC",
        "Valor",
        "Unidade de Medida",
        "Trimestre"
    ]
]

# Simplifica os nomes
df = df.rename(
    columns={
        "Unidade da Federação": "Estado",
        "Grupamentos ocupacionais no trabalho principal - PNADC": "Cargo",
        "Valor": "Salário médio",
        "Unidade de Medida": "Unidade",
        "Trimestre": "Período"
    }
)

# Converte salário para número
df["Salário médio"] = pd.to_numeric(
    df["Salário médio"],
    errors="coerce"
)

# Remove as linhas de total
df = df[df["Cargo"] != "Total"]

# Mostra uma amostra com 10 linhas
print(df.head(10).to_string(index=False))

  Estado                                                                                                Cargo  Salário médio Unidade           Período
Rondônia                                                                                 Diretores e gerentes         5520.0   Reais 2º trimestre 2026
Rondônia                                                            Profissionais das ciências e intelectuais         7087.0   Reais 2º trimestre 2026
Rondônia                                                              Técnicos e profissionais de nível médio         4812.0   Reais 2º trimestre 2026
Rondônia                                                                Trabalhadores de apoio administrativo         3125.0   Reais 2º trimestre 2026
Rondônia                                      Trabalhadores dos serviços, vendedores dos comércios e mercados         3503.0   Reais 2º trimestre 2026
Rondônia                           Trabalhadores qualificados da agropecuária, florestais, da 

In [4]:
df = df[df["Período"].str.contains("2022", na=False)]

In [5]:
import requests
import pandas as pd

url = "https://apisidra.ibge.gov.br/values/t/5444/n3/all/v/5932/p/all/c694/all"

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()

dados = resposta.json()

descricao_colunas = dados[0]

df = pd.DataFrame(dados[1:])

df = df.rename(columns=descricao_colunas)

df = df[
    [
        "Unidade da Federação",
        "Grupamentos ocupacionais no trabalho principal - PNADC",
        "Valor",
        "Unidade de Medida",
        "Trimestre"
    ]
]

df = df.rename(
    columns={
        "Unidade da Federação": "Estado",
        "Grupamentos ocupacionais no trabalho principal - PNADC": "Cargo",
        "Valor": "Salário médio",
        "Unidade de Medida": "Unidade",
        "Trimestre": "Período"
    }
)

df["Salário médio"] = pd.to_numeric(
    df["Salário médio"],
    errors="coerce"
)

df = df[df["Cargo"] != "Total"]

# Filtra somente 2022
df = df[df["Período"].str.contains("2022", na=False)]

# Mostra os 10 maiores salários
display(
    df.sort_values(
        "Salário médio",
        ascending=False
    ).head(10)
)

,Estado,Cargo,Salário médio,Unidade,Período
16031,Distrito Federal,Ocupações mal definidas,35541.0,Reais,4º trimestre 2022
16007,Distrito Federal,Ocupações mal definidas,23907.0,Reais,2º trimestre 2022
16009,Distrito Federal,Diretores e gerentes,14460.0,Reais,3º trimestre 2022
16021,Distrito Federal,Diretores e gerentes,14170.0,Reais,4º trimestre 2022
16030,Distrito Federal,"Membros das forças armadas, policiais e bombei...",11514.0,Reais,4º trimestre 2022
16018,Distrito Federal,"Membros das forças armadas, policiais e bombei...",11373.0,Reais,3º trimestre 2022
15997,Distrito Federal,Diretores e gerentes,11327.0,Reais,2º trimestre 2022
11209,Rio de Janeiro,Diretores e gerentes,10597.0,Reais,3º trimestre 2022
11821,São Paulo,Diretores e gerentes,10537.0,Reais,4º trimestre 2022
16006,Distrito Federal,"Membros das forças armadas, policiais e bombei...",10529.0,Reais,2º trimestre 2022


In [6]:
cargos_distintos = sorted(
    df["Cargo"].dropna().unique()
)

print(cargos_distintos)

['Diretores e gerentes', 'Membros das forças armadas, policiais e bombeiros militares', 'Ocupações elementares', 'Ocupações mal definidas', 'Operadores de instalações e máquinas e montadores', 'Profissionais das ciências e intelectuais', 'Trabalhadores de apoio administrativo', 'Trabalhadores dos serviços, vendedores dos comércios e mercados', 'Trabalhadores qualificados da agropecuária, florestais, da caça e da pesca', 'Trabalhadores qualificados, operários e artesãos da construção, das artes mecânicas e outros ofícios', 'Técnicos e profissionais de nível médio']


In [7]:
cargos_distintos = (
    df["Cargo"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

cargos_distintos = cargos_distintos.to_frame(name="Cargo")

display(cargos_distintos)

,Cargo
0,Diretores e gerentes
1,"Membros das forças armadas, policiais e bombei..."
2,Ocupações elementares
3,Ocupações mal definidas
4,Operadores de instalações e máquinas e montadores
5,Profissionais das ciências e intelectuais
6,Trabalhadores de apoio administrativo
7,"Trabalhadores dos serviços, vendedores dos com..."
8,"Trabalhadores qualificados da agropecuária, fl..."
9,"Trabalhadores qualificados, operários e artesã..."


In [8]:
cargos_excluir = [
    "Membros das forças armadas, policiais e bombeiros militares",
    "Ocupações elementares",
    "Operadores de instalações e máquinas e montadores",
    "Ocupações mal definidas",
    "Trabalhadores qualificados, operários e artesãos da construção, das artes mecânicas e outros ofícios"
    "Trabalhadores qualificados da agropecuária, florestais, da caça e da pesca"
]

df = df[
    ~df["Cargo"].astype(str).str.strip().isin(cargos_excluir)
].copy()

# Conferir os cargos restantes
display(df["Cargo"].drop_duplicates().sort_values())

397                                 Diretores e gerentes
398            Profissionais das ciências e intelectuais
400                Trabalhadores de apoio administrativo
401    Trabalhadores dos serviços, vendedores dos com...
402    Trabalhadores qualificados da agropecuária, fl...
403    Trabalhadores qualificados, operários e artesã...
399              Técnicos e profissionais de nível médio
Name: Cargo, dtype: str

In [9]:
# Mostrar as primeiras 20 linhas
display(df.head(20))

,Estado,Cargo,Salário médio,Unidade,Período
397,Rondônia,Diretores e gerentes,4984.0,Reais,2º trimestre 2022
398,Rondônia,Profissionais das ciências e intelectuais,4947.0,Reais,2º trimestre 2022
399,Rondônia,Técnicos e profissionais de nível médio,3350.0,Reais,2º trimestre 2022
400,Rondônia,Trabalhadores de apoio administrativo,2319.0,Reais,2º trimestre 2022
401,Rondônia,"Trabalhadores dos serviços, vendedores dos com...",2392.0,Reais,2º trimestre 2022
402,Rondônia,"Trabalhadores qualificados da agropecuária, fl...",2947.0,Reais,2º trimestre 2022
403,Rondônia,"Trabalhadores qualificados, operários e artesã...",2352.0,Reais,2º trimestre 2022
409,Rondônia,Diretores e gerentes,4980.0,Reais,3º trimestre 2022
410,Rondônia,Profissionais das ciências e intelectuais,5031.0,Reais,3º trimestre 2022
411,Rondônia,Técnicos e profissionais de nível médio,4158.0,Reais,3º trimestre 2022


In [10]:
# Garantir que a coluna de salário esteja no formato numérico
df["Salário médio"] = pd.to_numeric(
    df["Salário médio"],
    errors="coerce"
)

# Criar a coluna de salário em dólar
df["Salário em dólar"] = (
    df["Salário médio"] / 5.19
).round(2)

# Visualizar o resultado
display(
    df[["Salário médio", "Salário em dólar"]].head(20)
)

,Salário médio,Salário em dólar
397,4984.0,960.31
398,4947.0,953.18
399,3350.0,645.47
400,2319.0,446.82
401,2392.0,460.89
402,2947.0,567.82
403,2352.0,453.18
409,4980.0,959.54
410,5031.0,969.36
411,4158.0,801.16


In [11]:
# Mostrar as primeiras 20 linhas
display(df.head(20))

,Estado,Cargo,Salário médio,Unidade,Período,Salário em dólar
397,Rondônia,Diretores e gerentes,4984.0,Reais,2º trimestre 2022,960.31
398,Rondônia,Profissionais das ciências e intelectuais,4947.0,Reais,2º trimestre 2022,953.18
399,Rondônia,Técnicos e profissionais de nível médio,3350.0,Reais,2º trimestre 2022,645.47
400,Rondônia,Trabalhadores de apoio administrativo,2319.0,Reais,2º trimestre 2022,446.82
401,Rondônia,"Trabalhadores dos serviços, vendedores dos com...",2392.0,Reais,2º trimestre 2022,460.89
402,Rondônia,"Trabalhadores qualificados da agropecuária, fl...",2947.0,Reais,2º trimestre 2022,567.82
403,Rondônia,"Trabalhadores qualificados, operários e artesã...",2352.0,Reais,2º trimestre 2022,453.18
409,Rondônia,Diretores e gerentes,4980.0,Reais,3º trimestre 2022,959.54
410,Rondônia,Profissionais das ciências e intelectuais,5031.0,Reais,3º trimestre 2022,969.36
411,Rondônia,Técnicos e profissionais de nível médio,4158.0,Reais,3º trimestre 2022,801.16


In [12]:
import numpy as np

# Permite repetir sempre a mesma distribuição
gerador = np.random.default_rng(42)

correspondencia_cargos = {
    "Diretores e gerentes": [
        "Manager",
        "Manufacturing Director",
        "Research Director"
    ],

    "Profissionais das ciências e intelectuais": [
        "Research Scientist"
    ],

    "Técnicos e profissionais de nível médio": [
        "Healthcare Representative",
        "Laboratory Technician"
    ],

    "Trabalhadores de apoio administrativo": [
        "Human Resources"
    ],

    "Trabalhadores dos serviços, vendedores dos comércios e mercados": [
        "Sales Executive",
        "Sales Representative"
    ]

}

def definir_cargo_ingles(cargo):
    cargo = str(cargo).strip()
    opcoes = correspondencia_cargos.get(cargo)

    if opcoes:
        return gerador.choice(opcoes)

    return "Não classificado"

df["Cargo em inglês"] = df["Cargo"].apply(definir_cargo_ingles)

display(
    df[["Cargo", "Cargo em inglês"]].head(30)
)

,Cargo,Cargo em inglês
397,Diretores e gerentes,Manager
398,Profissionais das ciências e intelectuais,Research Scientist
399,Técnicos e profissionais de nível médio,Laboratory Technician
400,Trabalhadores de apoio administrativo,Human Resources
401,"Trabalhadores dos serviços, vendedores dos com...",Sales Representative
402,"Trabalhadores qualificados da agropecuária, fl...",Não classificado
403,"Trabalhadores qualificados, operários e artesã...",Não classificado
409,Diretores e gerentes,Manufacturing Director
410,Profissionais das ciências e intelectuais,Research Scientist
411,Técnicos e profissionais de nível médio,Healthcare Representative


In [13]:
cargos_unicos = sorted(df["Cargo"].dropna().unique())

In [14]:
df_dim_cargo = pd.DataFrame(cargos_unicos, columns=["Cargo"])
df_dim_cargo.insert(0, "id_cargo", range(1, len(df_dim_cargo) + 1))
print("Tabela Dimensão de Cargos criada:")
display(df_dim_cargo)

Tabela Dimensão de Cargos criada:


,id_cargo,Cargo
0,1,Diretores e gerentes
1,2,Profissionais das ciências e intelectuais
2,3,Trabalhadores de apoio administrativo
3,4,"Trabalhadores dos serviços, vendedores dos com..."
4,5,"Trabalhadores qualificados da agropecuária, fl..."
5,6,"Trabalhadores qualificados, operários e artesã..."
6,7,Técnicos e profissionais de nível médio


In [15]:
df = df.merge(df_dim_cargo, on="Cargo", how="left")

In [16]:
df = df.drop(columns=["Cargo"])
print("\nComo ficou a Tabela Principal do IBGE (agora com o id_cargo):")
display(df[["Estado", "Salário médio", "id_cargo", "Salário em dólar"]].head())


Como ficou a Tabela Principal do IBGE (agora com o id_cargo):


,Estado,Salário médio,id_cargo,Salário em dólar
0,Rondônia,4984.0,1,960.31
1,Rondônia,4947.0,2,953.18
2,Rondônia,3350.0,7,645.47
3,Rondônia,2319.0,3,446.82
4,Rondônia,2392.0,4,460.89


# IMPORTAÇÃO DE BASE IBM

In [17]:
# Instalar a biblioteca oficial do Kaggle

!pip install -q kagglehub


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import kagglehub
import pandas as pd
import os
import glob

# Baixar a base diretamente do Kaggle
caminho = kagglehub.dataset_download(
    "pavansubhasht/ibm-hr-analytics-attrition-dataset"
)

# Localizar o arquivo CSV baixado
arquivos_csv = glob.glob(
    os.path.join(caminho, "*.csv")
)

print("Arquivos encontrados:", arquivos_csv)

# Importar o primeiro CSV encontrado
df = pd.read_csv(arquivos_csv[0])

# Mostrar as 5 primeiras linhas
df.head()

C:\Users\82424919\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 50.1k/50.1k [00:00<00:00, 374kB/s]

Extracting files...
Arquivos encontrados: ['C:\\Users\\82424919\\.cache\\kagglehub\\datasets\\pavansubhasht\\ibm-hr-analytics-attrition-dataset\\versions\\1\\WA_Fn-UseC_-HR-Employee-Attrition.csv']


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [19]:
# Criar faixas salariais

limites = [
    0,
    3000,
    5000,
    8000,
    12000,
    float("inf")
]

nomes_faixas = [
    "Até 3.000",
    "De 3.001 a 5.000",
    "De 5.001 a 8.000",
    "De 8.001 a 12.000",
    "Acima de 12.000"
]

df["FaixaSalarial"] = pd.cut(
    df["MonthlyIncome"],
    bins=limites,
    labels=nomes_faixas,
    include_lowest=True
)

display(
    df[
        ["EmployeeNumber", "MonthlyIncome", "FaixaSalarial"]
    ].head(10)
)

,EmployeeNumber,MonthlyIncome,FaixaSalarial
0,1,5993,De 5.001 a 8.000
1,2,5130,De 5.001 a 8.000
2,4,2090,Até 3.000
3,5,2909,Até 3.000
4,7,3468,De 3.001 a 5.000
5,8,3068,De 3.001 a 5.000
6,10,2670,Até 3.000
7,11,2693,Até 3.000
8,12,9526,De 8.001 a 12.000
9,13,5237,De 5.001 a 8.000


In [20]:
distribuicao_salarial = (
    df["FaixaSalarial"]
    .value_counts()
    .sort_index()
    .rename_axis("Faixa salarial")
    .reset_index(name="Quantidade de funcionários")
)

display(distribuicao_salarial)

,Faixa salarial,Quantidade de funcionários
0,Até 3.000,395
1,De 3.001 a 5.000,354
2,De 5.001 a 8.000,340
3,De 8.001 a 12.000,186
4,Acima de 12.000,195


In [21]:
faixas = [17, 25, 35, 45, 55, 65]
nomes_faixas = ["18-25", "26-35", "36-45", "46-55", "56-65"]

df["Faixa Etária"] = pd.cut(
    df["Age"],
    bins=faixas,
    labels=nomes_faixas
)

distribuicao_idade = (
    df["Faixa Etária"]
    .value_counts()
    .sort_index()
    .rename_axis("Faixa Etária")
    .reset_index(name="Quantidade")
)

distribuicao_idade["Percentual"] = (
    distribuicao_idade["Quantidade"] / len(df) * 100
).round(2)

display(distribuicao_idade)

,Faixa Etária,Quantidade,Percentual
0,18-25,123,8.37
1,26-35,606,41.22
2,36-45,468,31.84
3,46-55,226,15.37
4,56-65,47,3.20


In [22]:
tabela_colunas = pd.DataFrame({
    "Número": range(1, len(df.columns) + 1),
    "Nome da coluna": df.columns
})

display(tabela_colunas)

,Número,Nome da coluna
0,1,Age
1,2,Attrition
2,3,BusinessTravel
3,4,DailyRate
4,5,Department
5,6,DistanceFromHome
6,7,Education
7,8,EducationField
8,9,EmployeeCount
9,10,EmployeeNumber


In [23]:
df = df.drop(columns=["Over18"])
print(f"Quantidade de colunas: {df.shape[1]}")
display(df.head())

Quantidade de colunas: 36


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,FaixaSalarial,Faixa Etária
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,0,8,0,1,6,4,0,5,De 5.001 a 8.000,36-45
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,1,10,3,3,10,7,1,7,De 5.001 a 8.000,46-55
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,0,7,3,3,0,0,0,0,Até 3.000,36-45
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,0,8,3,3,8,7,3,0,Até 3.000,26-35
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,1,6,3,3,2,2,2,2,De 3.001 a 5.000,26-35


In [24]:
valores_distintos = sorted(
    df["EmployeeCount"].dropna().unique()
)

print(valores_distintos)

[np.int64(1)]


In [25]:
df = df.drop(columns=["StockOptionLevel"])
print(f"Quantidade de colunas: {df.shape[1]}")
display(df.head())

Quantidade de colunas: 35


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,FaixaSalarial,Faixa Etária
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,80,8,0,1,6,4,0,5,De 5.001 a 8.000,36-45
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,80,10,3,3,10,7,1,7,De 5.001 a 8.000,46-55
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,80,7,3,3,0,0,0,0,Até 3.000,36-45
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,80,8,3,3,8,7,3,0,Até 3.000,26-35
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,80,6,3,3,2,2,2,2,De 3.001 a 5.000,26-35


In [26]:
valores_distintos = sorted(
    df["Education"].dropna().unique()
)

print(valores_distintos)

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [27]:
quantidade_linhas = df.shape[0]
quantidade_colunas = df.shape[1]
linhas_duplicadas = df.duplicated().sum()

print("AVALIAÇÃO DA BASE")
print("-" * 40)
print(f"Quantidade de linhas: {quantidade_linhas}")
print(f"Quantidade de colunas: {quantidade_colunas}")
print(f"Linhas duplicadas: {linhas_duplicadas}")

AVALIAÇÃO DA BASE
----------------------------------------
Quantidade de linhas: 1470
Quantidade de colunas: 35
Linhas duplicadas: 0


In [28]:
cargos_unicos_ibm = sorted(df["JobRole"].dropna().unique())
df_dim_cargo_ibm = pd.DataFrame(cargos_unicos_ibm, columns=["JobRole"])
df_dim_cargo_ibm.insert(0, "id_cargo", range(1, len(df_dim_cargo_ibm) + 1))
print("Tabela Dimensão de Cargos (IBM) criada:")
display(df_dim_cargo_ibm)

Tabela Dimensão de Cargos (IBM) criada:


,id_cargo,JobRole
0,1,Healthcare Representative
1,2,Human Resources
2,3,Laboratory Technician
3,4,Manager
4,5,Manufacturing Director
5,6,Research Director
6,7,Research Scientist
7,8,Sales Executive
8,9,Sales Representative


In [29]:
df = df.merge(df_dim_cargo_ibm, on="JobRole", how="left")
df = df.drop(columns=["JobRole"])
print("\nComo ficou a Tabela Principal da IBM (agora com o id_cargo):")
display(df[["EmployeeNumber", "Department", "id_cargo", "MonthlyIncome","Attrition"]].head())


Como ficou a Tabela Principal da IBM (agora com o id_cargo):


,EmployeeNumber,Department,id_cargo,MonthlyIncome,Attrition
0,1,Sales,8,5993,Yes
1,2,Research & Development,7,5130,No
2,4,Research & Development,3,2090,Yes
3,5,Research & Development,7,2909,No
4,7,Research & Development,3,3468,No


# TABELAS Qualitativas
Será necessario puxar uma tabela terceira de pesquisa por paises

1.   Item da lista
2.   Item da lista 

In [32]:
import pandas as pd

# Define o nome do arquivo (ele precisa estar na mesma pasta do seu código no VS Code)
nome_arquivo = "db_a3_engenharia_de_dados.xlsx"

# Lê as duas abas do Excel diretamente do seu computador
df_custo = pd.read_excel(nome_arquivo, sheet_name="ranking_custo_trabalho")
df_salario = pd.read_excel(nome_arquivo, sheet_name="ranking_salario")

# Mostra as primeiras linhas para confirmar que deu tudo certo
print("ranking_custo_trabalho")
display(df_custo.head())

print("\nranking_salario")
display(df_salario.head())

ranking_custo_trabalho


,id_pais,pais,ultimo,anterior,referencia,unidade
0,1,Alemanha,124,124,2022,Pontos
1,2,Austrália,109,108,2022,Pontos
2,3,Áustria,132,148,2022,Pontos
3,4,Bélgica,120,119,2022,Pontos
4,5,Brasil,166,176,2022,Pontos



ranking_salario


,id_pais,Pais,ultimo,anterior,referencia,unidade
0,1,Alemanha,76285,74854,2022,USD
1,2,Austrália,72018,71238,2022,USD
2,3,Áustria,78301,77703,2022,USD
3,4,Bélgica,80009,78614,2022,USD
4,5,Brasil,24463,24140,2022,USD
